# Quick Start with AutoDDG

This notebook demonstrates how to generate dataset descriptions, expand them for search, and evaluate their quality using **AutoDDG**.

AutoDDG supports both **API-based** (OpenAI) and **local LLM** (transformers) modes.

---

## 1. Imports

In [ ]:
import pandas as pd
from openai import OpenAI
import sys

from autoddg import AutoDDG, GPTEvaluator
from autoddg.utils import get_sample

## 2. Load Dataset

First, let's load and prepare the dataset. This step is common for both OpenAI API and local LLM modes.

In [ ]:
# Load dataset
csv_file = "clark_dataset.csv"
title = "Renal Cell Carcinoma"
original_description = (
    "This study reports a large-scale proteogenomic analysis of ccRCC to discern the functional impact "
    "of genomic alterations and provides evidence for rational treatment selection stemming from ccRCC pathobiology"
)
csv_df = pd.read_csv(csv_file)

# Sample rows for processing
sample_df, dataset_sample = get_sample(csv_df, sample_size=100)

## 3. Initialize AutoDDG

Choose one of the following initialization options:

### Option A: Using OpenAI API (Recommended for quick start)


In [ ]:
# Option A: OpenAI API
my_api_key = "YOUR_OPENAI_API_KEY"  # Replace with your key
client = OpenAI(api_key=my_api_key)
model_name = "gpt-4o-mini"

# Initialize AutoDDG with API client
auto_ddg = AutoDDG(client=client, model_name=model_name)


### Option B: Using Local LLM (Qwen, Llama, etc.)

For local LLM support, install the optional dependencies first:
```bash
pip install autoddg[local-llm]
# or
pip install transformers torch
```

In [ ]:
# Option B: Local LLM (uncomment to use)
import torch
# 
# # Initialize AutoDDG with local LLM
auto_ddg = AutoDDG(
    client=None,
    model_name="Qwen/Qwen3-4B-Instruct-2507",  # or any HuggingFace model
    use_local_llm=True,
    local_llm_device="cuda",  # or "cpu" if no GPU
    local_llm_dtype="bfloat16",  # or "float16", "float32"
)

## 4. Prepare Context

After initializing AutoDDG, we can generate profiles and topic:

In [ ]:
# Generate basic structural profile
basic_profile, structural_profile = auto_ddg.profile_dataframe(csv_df)

# Generate topic
data_topic = auto_ddg.generate_topic(
    title=title,
    original_description=original_description,
    dataset_sample=dataset_sample,
)

## 5. Semantic Analysis - Processing Modes

AutoDDG provides four processing modes for semantic analysis. Choose the one that best fits your use case:

| Mode | OpenAI API | Local LLM | Description |
|------|------------|-----------|-------------|
| **Sequential** | ✅ | ✅ | Default mode, processes columns one by one |
| **Multi-threading** | ✅ | ❌ | Parallel processing for faster execution |
| **Group-prompting** | ✅ | ✅ | Processes multiple columns in one prompt |
| **Batch processing** | ❌ | ✅ | Efficient GPU utilization for local models |

### 5.1 Sequential Mode (Default)

**Works with:** Both OpenAI API and Local LLM

Processes columns one by one sequentially. This is the default mode and works with both API and local LLMs.

In [ ]:
# Sequential mode (default) - works with both OpenAI API and Local LLM
semantic_profile_details = auto_ddg.analyze_semantics(sample_df)

# Combine with structural profile
semantic_profile = "\n".join(
    section for section in [structural_profile, semantic_profile_details] if section
)

print("Semantic analysis completed (sequential mode)")

### 5.2 Multi-threading Mode (OpenAI API Only)

**Works with:** OpenAI API only

Uses multi-threading to process columns in parallel for faster execution. This mode is only available for OpenAI API clients.

In [ ]:
# Multi-threading mode (only works with OpenAI API)
# Uncomment and use this if you initialized with OpenAI API (Option A)

# semantic_profile_details = auto_ddg.analyze_semantics(
#     sample_df,
#     use_multi_threading=True,
#     max_workers=8,  # Optional: number of parallel workers (default: min(32, num_columns))
# )
# # 
# semantic_profile = "\n".join(
#     section for section in [structural_profile, semantic_profile_details] if section
# )

# print("Semantic analysis completed (multi-threading mode)")

### 5.3 Group-prompting Mode (Both API and Local LLM)

**Works with:** Both OpenAI API and Local LLM

Processes multiple columns in a single prompt to reduce API calls. This is efficient for both API and local LLMs.

In [ ]:
# Group-prompting mode: process all columns in one call
# Works with both OpenAI API and Local LLM

# # Option 1: Process all columns at once (most efficient)
# semantic_profile_details = auto_ddg.analyze_semantics(
#     sample_df,
#     use_group_prompting=True,
#     group_size=0,  # 0 = all columns at once
# )

# # Option 2: Process in groups of specified size
# semantic_profile_details = auto_ddg.analyze_semantics(
#     sample_df,
#     use_group_prompting=True,
#     group_size=5,  # Process in groups of 5 columns
# )

# semantic_profile = "\n".join(
#     section for section in [structural_profile, semantic_profile_details] if section
# )

# print("Semantic analysis completed (group-prompting mode)")


### 5.4 Batch Processing Mode (Local LLM Only)

**Works with:** Local LLM only

Processes columns in batches using batch inference for efficient GPU utilization. This mode is only available for local LLMs.


In [ ]:
# Batch processing mode (only works with local LLMs)
# Uncomment and use this if you initialized with Local LLM (Option B)

# semantic_profile_details = auto_ddg.analyze_semantics(
#     sample_df,
#     use_batch_processing=True,
#     batch_size=32,  # Number of columns to process per batch
# )

# semantic_profile = "\n".join(
#     section for section in [structural_profile, semantic_profile_details] if section
# )

# print("Semantic analysis completed (batch processing mode)")


## 6. Generate Descriptions

Now we create both a **general dataset description** and a **search-focused description** using the semantic profile we generated.


In [ ]:
# General description
prompt, description = auto_ddg.describe_dataset(
    dataset_sample=dataset_sample,
    dataset_profile=basic_profile,
    use_profile=True,
    semantic_profile=semantic_profile,
    use_semantic_profile=True,
    data_topic=data_topic,
    use_topic=True,
)

# Search-focused description
search_prompt, search_focused_description = auto_ddg.expand_description_for_search(
    description=description,
    topic=data_topic,
)


### 6.1 General Description


In [ ]:
print(description)


### 6.2 Search-Focused Description


In [ ]:
print(search_focused_description)


## 7. Evaluate Quality (Optional)

Finally, we can use the evaluator to score both descriptions. **Note:** Evaluation currently requires OpenAI API access.


In [ ]:
# Attach evaluator (requires OpenAI API key)
# Note: Evaluation currently requires API access
try:
    auto_ddg.set_evaluator(GPTEvaluator(gpt4_api_key=my_api_key))
    
    # Score descriptions
    general_score = auto_ddg.evaluate_description(description)
    search_score = auto_ddg.evaluate_description(search_focused_description)
    
    print("Score of the general description:", general_score)
    print("Score of the search-focused description:", search_score)
except Exception as e:
    print(f"Evaluation skipped: {e}")
    print("Note: Evaluation requires OpenAI API access")
